In [ ]:
import pathlib
from pathlib import Path
import pandas as pd
import numpy as np
import time
import warnings
import os

# Configuração para exibir todas as colunas no pandas (útil para inspeção)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO DE CAMINHOS ---
CURRENT_PATH = Path.cwd()

if CURRENT_PATH.name == 'notebooks':
    PROJECT_ROOT = CURRENT_PATH.parent
else:
    PROJECT_ROOT = CURRENT_PATH

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(exist_ok=True, parents=True)

print(f"Raiz do Projeto: {PROJECT_ROOT}")
print(f"Diretório de Dados Brutos: {RAW_DIR}")

In [ ]:
# --- CÉLULA 2: Identificar os Arquivos Principais por Tamanho ---

print("Varrendo diretório em busca dos maiores arquivos CSV...")

csv_files = []
for file_path in RAW_DIR.rglob("*.csv"):
    # Obtém o tamanho do arquivo em MB
    size_mb = file_path.stat().st_size / (1024 * 1024)
    csv_files.append({
        "path": file_path,
        "name": file_path.name,
        "size_mb": size_mb,
        "folder": file_path.parent.name
    })

# Cria um DataFrame para visualizar melhor
df_files = pd.DataFrame(csv_files)

if not df_files.empty:
    # Ordena pelos maiores
    df_files = df_files.sort_values(by="size_mb", ascending=False)
    
    print(f"\nTop 10 Maiores Arquivos Encontrados:")
    print("-" * 100)
    print(f"{'Nome do Arquivo':<40} | {'Pasta':<30} | {'Tamanho (MB)':>15}")
    print("-" * 100)
    
    # Mostra os top 10 para confirmarmos se os 5 gigantes estão no topo
    for _, row in df_files.head(10).iterrows():
        print(f"{row['name']:<40} | {row['folder']:<30} | {row['size_mb']:>15.2f} MB")
        
    # Armazena os caminhos dos 5 maiores para usarmos depois
    # Assumindo que os 5 maiores são os que queremos
    target_files = df_files.head(5)['path'].tolist()
else:
    print("Nenhum arquivo CSV encontrado.")

In [ ]:
# --- CÉLULA 3: Inspeção do Formato dos Arquivos Principais ---

if 'target_files' in locals() and target_files:
    # Vamos pegar o MAIOR arquivo para inspecionar
    big_file = target_files[0]
    
    print(f"Inspecionando arquivo: {big_file.name}")
    print(f"Caminho: {big_file}")
    
    # Ler as primeiras 5 linhas como texto puro primeiro para ver o separador
    print("\n--- Primeiras 3 linhas (Texto Bruto) ---")
    with open(big_file, 'r', encoding='utf-8', errors='replace') as f:
        for _ in range(3):
            print(f.readline().strip())
            
    print("\n--- Tentativa de Leitura com Pandas (Separador ',') ---")
    try:
        df_sample = pd.read_csv(big_file, nrows=5, sep=',') # Tenta vírgula primeiro
        display(df_sample.head())
        print("\nColunas detectadas:", list(df_sample.columns))
        
        # Verifica coluna de tempo
        time_cols = [c for c in df_sample.columns if 'time' in c.lower() or 'tempo' in c.lower()]
        if time_cols:
            print(f"\nColuna de Tempo provável: {time_cols}")
            print(f"Exemplo de valor: {df_sample[time_cols[0]].iloc[0]}")
    except Exception as e:
        print(f"Erro ao ler com vírgula: {e}")

else:
    print("Variável 'target_files' não definida. Rode a Célula 2.")

In [ ]:
# -----------------------------------------------------------------------------
# CÉLULA 3 (ATUALIZADA): DICIONÁRIO CANÔNICO COM CAMPO E ATIVIDADE
# -----------------------------------------------------------------------------

CANON = {
    # --- TEMPO E MOTOR ---
    "Time": "timestamp",
    "Time_(s)": "timestamp",
    "EngSpeed": "engine_rpm",
    "EngSpeed_(RPM)": "engine_rpm",
    "EngFuelRate": "fuel_rate",
    "EngFuelRate_(L/h)": "fuel_rate",
    
    # --- VELOCIDADE ---
    "WheelBasedMachineSpeed": "veh_speed",
    "WheelBasedMachineSpeed_(m/s)": "veh_speed",
    "SpeedOverGround_(m/s)": "veh_speed",

    # --- LOCALIZAÇÃO ---
    "Latitude": "latitude",
    "Latitude_(°)": "latitude",
    "Longitude": "longitude",
    "Longitude_(°)": "longitude",
    
    # --- CARGA E ENGATE ---
    "ActualEngPercentTorque": "engine_torque",
    "ActualEngPercentTorque_(%)": "engine_torque",
    "RearPTOOutputShaftSpeed": "pto_rpm",
    "RearPTOOutputShaftSpeed_(RPM)": "pto_rpm",
    "RearHitchPosition": "hitch_pos",
    "RearHitchPosition_[-]": "hitch_pos",
    "FrontHitchPos": "front_hitch_pos",
    "FrontHitchPos_(%)": "front_hitch_pos",
    
    # --- METADADOS / FEATURE ENGINEERING (Novos) ---
    "Cluster_[-]": "cluster_id",   # Essencial para criar o 'field'
    "Cluster": "cluster_id",
    "WorkType_[-]": "activity_raw", # Atividade bruta
    "WorkType": "activity_raw",
    "Status_[-]": "status",
    "Status": "status"
}

print("Dicionário CANON atualizado com suporte a Cluster e Atividade.")

In [6]:
# ==============================================================================
# CÉLULA 4 (REFINADA): PROCESSAMENTO COM ENGENHARIA DE FEATURES
# ==============================================================================

import pandas as pd
import numpy as np
import time
from pathlib import Path

def clean_and_standardize(df):
    """
    Limpa, padroniza e CRIA as colunas field e activity a partir dos dados brutos.
    """
    # 1. Renomear (Aplica o CANON)
    df = df.rename(columns=CANON)
    
    # 1.5. Tratar colunas duplicadas (timestamp)
    if not df.columns.is_unique:
        if 'timestamp' in df.columns and isinstance(df['timestamp'], pd.DataFrame):
            ts_combined = df['timestamp'].bfill(axis=1).iloc[:, 0]
            df = df.drop(columns=['timestamp'])
            df['timestamp'] = ts_combined
        df = df.loc[:, ~df.columns.duplicated()]

    # 2. Filtrar colunas de interesse (agora inclui cluster_id e activity_raw)
    wanted_cols = list(set(CANON.values()))
    # Garante que 'tractor' e 'source_file' não sejam removidos se já existirem
    for meta in ['tractor', 'source_file']:
        if meta in df.columns: wanted_cols.append(meta)
            
    existing_cols = [c for c in wanted_cols if c in df.columns]
    df = df[existing_cols].copy()
    
    # --- ENGENHARIA DE FEATURES (NOVO) ---
    
    # A) Criar 'field' a partir de 'cluster_id'
    if 'cluster_id' in df.columns:
        # Preenche nulos com -1 e converte para int
        df['cluster_id'] = pd.to_numeric(df['cluster_id'], errors='coerce').fillna(-1).astype(int)
        
        # Cria a string "Field_X" (ignorando clusters inválidos se necessário)
        # Cluster 0 muitas vezes é transporte/estrada, mas vamos mapear tudo.
        df['field'] = 'Field_' + df['cluster_id'].astype(str)
        
        # Opcional: Remover o cluster_id bruto para economizar espaço
        # df.drop(columns=['cluster_id'], inplace=True)
    else:
        df['field'] = 'Unknown'

    # B) Padronizar 'activity'
    if 'activity_raw' in df.columns:
        df['activity'] = df['activity_raw'].fillna('General').astype(str)
        df.drop(columns=['activity_raw'], inplace=True)
    else:
        df['activity'] = 'General'

    # 3. Conversão de Timestamp
    if 'timestamp' in df.columns:
        if df['timestamp'].dtype == object:
            try:
                df['timestamp'] = pd.to_timedelta(df['timestamp']).dt.total_seconds()
            except Exception:
                df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    
    # 4. Garantir tipos numéricos
    numeric_cols = ['engine_rpm', 'fuel_rate', 'veh_speed', 'latitude', 'longitude', 
                    'pto_rpm', 'engine_torque', 'hitch_pos']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    return df

def process_large_csv_to_parquet(csv_path, output_path, chunk_size=500000):
    print(f"Processando: {csv_path.name} -> {output_path.name}")
    
    temp_files = []
    chunk_count = 0
    total_rows = 0
    
    try:
        # Lê o CSV em blocos
        with pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False) as reader:
            for i, chunk in enumerate(reader):
                df_clean = clean_and_standardize(chunk)
                
                # Metadados de Arquivo (Trator vem do nome do arquivo)
                if 'tractor' not in df_clean.columns:
                    # Remove extensão e sufixos comuns
                    clean_name = csv_path.stem.replace('_processed', '').split('.')[0]
                    df_clean['tractor'] = clean_name
                
                df_clean['source_file'] = csv_path.name
                df_clean.dropna(subset=['timestamp'], inplace=True)
                
                if not df_clean.empty:
                    temp_file = output_path.parent / f"temp_{output_path.stem}_{i}.parquet"
                    df_clean.to_parquet(temp_file, index=False)
                    temp_files.append(temp_file)
                    total_rows += len(df_clean)
                    chunk_count += 1
                    
                    if i % 10 == 0 and i > 0:
                        print(f"   ...bloco {i} ({total_rows} linhas acumuladas)")

        print(f"Consolidando {chunk_count} blocos...")
        
        if temp_files:
            full_df = pd.concat([pd.read_parquet(f) for f in temp_files], ignore_index=True)
            
            if 'timestamp' in full_df.columns:
                full_df.sort_values(by=['timestamp'], inplace=True)
            
            # Reordenar colunas para ficar bonito
            cols_order = ['timestamp', 'tractor', 'activity', 'field', 'fuel_rate', 'veh_speed', 'engine_rpm']
            existing_order = [c for c in cols_order if c in full_df.columns]
            remaining = [c for c in full_df.columns if c not in cols_order]
            full_df = full_df[existing_order + remaining]

            full_df.to_parquet(output_path, index=False)
            
            for f in temp_files: f.unlink()
                
            print(f"Sucesso! {output_path.name} com {total_rows:,} registros e colunas: {list(full_df.columns[:5])}...")
            return True
        else:
            return False

    except Exception as e:
        print(f"ERRO em {csv_path.name}: {e}")
        for f in temp_files:
            if f.exists(): f.unlink()
        return False

In [ ]:
# ==============================================================================
# CÉLULA 5: EXECUÇÃO DO PIPELINE PARA OS 5 ARQUIVOS PRINCIPAIS
# ==============================================================================

# Lista manual dos 5 arquivos que identificamos como os principais
# Ajuste os nomes das pastas se necessário, baseando-se no output da Célula 2
target_files_names = [
    "Fendt 722.csv",
    "Fendt 314.csv",
    "Fendt 724.csv",
    "Fendt 820.csv",
    "Fendt 211.csv"
]

print(f"Iniciando pipeline para {len(target_files_names)} arquivos alvo...")
start_global = time.time()

# Encontra os caminhos completos
files_to_process = []
for fname in target_files_names:
    found = list(RAW_DIR.rglob(fname))
    if found:
        # Pega o primeiro encontrado (geralmente o correto na estrutura)
        files_to_process.append(found[0])
    else:
        print(f"ALERTA: Arquivo {fname} não encontrado no diretório RAW.")

# Processa um por um (sequencial para não estourar memória com arquivos de 3GB)
for csv_path in files_to_process:
    output_name = csv_path.stem + "_processed.parquet"
    output_path = PROC_DIR / output_name
    
    # Chama a função de processamento em chunks
    success = process_large_csv_to_parquet(csv_path, output_path)
    
    if success:
        print(f"-> Concluído: {output_name}\n")
    else:
        print(f"-> Falha: {output_name}\n")

print(f"Pipeline finalizado em {time.time() - start_global:.2f} segundos.")

In [ ]:
# ==============================================================================
# CÉLULA 6: VALIDAÇÃO DE CONSISTÊNCIA E INSPEÇÃO FINAL
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Caminho do arquivo consolidado final (gerado na Célula 4 ou 5)
# Se você gerou arquivos separados na Célula 5, precisará consolidá-los primeiro ou ler um deles.
# Assumindo que você quer validar os arquivos processados individuais ou um consolidado.

# Vamos verificar os arquivos gerados na pasta processed
processed_files = list(PROC_DIR.glob("*_processed.parquet"))

if not processed_files:
    print("Nenhum arquivo processado encontrado para validação.")
else:
    print(f"Encontrados {len(processed_files)} arquivos processados. Validando amostra e tipos...")
    
    # Lista para armazenar estatísticas de cada arquivo
    validation_stats = []
    
    for p_file in processed_files:
        try:
            df = pd.read_parquet(p_file)
            
            # 1. Verificação de Tipos na coluna Timestamp
            ts_types = df['timestamp'].apply(type).unique()
            ts_is_numeric = pd.api.types.is_numeric_dtype(df['timestamp'])
            
            # 2. Contagem de Nulos
            nulos = df['timestamp'].isnull().sum()
            
            # 3. Estatísticas básicas
            min_ts = df['timestamp'].min()
            max_ts = df['timestamp'].max()
            duration = max_ts - min_ts if pd.notnull(min_ts) and pd.notnull(max_ts) else 0
            
            validation_stats.append({
                'Arquivo': p_file.name,
                'Total Linhas': len(df),
                'Tipos Timestamp': str(ts_types),
                'É Numérico?': ts_is_numeric,
                'Nulos': nulos,
                'Duração (s)': duration
            })
            
            # Mostra o HEAD do primeiro arquivo apenas para inspeção visual
            if p_file == processed_files[0]:
                print(f"\n--- Amostra do DataFrame: {p_file.name} ---")
                display(df.head())
                print("-" * 80)

        except Exception as e:
            print(f"Erro ao ler {p_file.name}: {e}")

    # Exibe o relatório de validação cruzada
    if validation_stats:
        df_stats = pd.DataFrame(validation_stats)
        print("\n--- Relatório de Validação Cruzada (Timestamp) ---")
        # Se 'É Numérico?' for True para todos, passou no teste
        if df_stats['É Numérico?'].all():
            print("✅ SUCESSO: A coluna 'timestamp' é numérica em TODOS os arquivos.")
        else:
            print("❌ ALERTA: Existem arquivos com 'timestamp' não numérico!")
            
        display(df_stats)

In [ ]:
# ==============================================================================
# CÉLULA 7: CONSOLIDAÇÃO DOS DADOS DE TELEMETRIA
# ==============================================================================

import pandas as pd
import time
from pathlib import Path

# Configuração de Caminhos
CURATED_DIR = PROJECT_ROOT / "data" / "curated"
CURATED_DIR.mkdir(exist_ok=True, parents=True)

# Lista dos arquivos processados na etapa anterior
processed_files = list(PROC_DIR.glob("*_processed.parquet"))

if not processed_files:
    print("ERRO: Nenhum arquivo processado encontrado para consolidação.")
else:
    print(f"Iniciando consolidação de {len(processed_files)} arquivos...")
    start_time = time.time()

    # Leitura e Concatenação
    # Como os arquivos já estão em Parquet e limpos, isso é muito rápido
    df_list = []
    for p_file in processed_files:
        print(f"  Carregando: {p_file.name}")
        df = pd.read_parquet(p_file)
        df_list.append(df)
    
    full_df = pd.concat(df_list, ignore_index=True)
    
    # Ordenação Crítica para Séries Temporais
    # Ordenamos por Trator e Tempo para garantir que o cálculo de delta_t funcione
    print("Ordenando dataset consolidado...")
    full_df.sort_values(by=['tractor', 'timestamp'], inplace=True)
    
    # Reset do índice para limpeza
    full_df.reset_index(drop=True, inplace=True)

    print(f"Consolidação concluída em {time.time() - start_time:.2f}s")
    print(f"Total de Registros: {len(full_df):,}")
    print(f"Tratores presentes: {full_df['tractor'].unique()}")

In [ ]:
# Certifique-se de que o 'full_df' está carregado
if 'full_df' not in locals():
    print("Atenção: Rode as células anteriores para carregar full_df.")

import plotly.express as px
import pandas as pd

print("\n--- INÍCIO DA ANÁLISE ESTATÍSTICA EXPLORATÓRIA (EDA) APROFUNDADA ---\n")

# --- CORREÇÃO DA AMOSTRAGEM ---
# Seus dados são 10Hz (10 pontos por segundo).
# Logo, para ter 1 hora, precisamos de 3600 segundos * 10 pontos = 36.000 pontos.
POINTS_PER_SECOND = 10 
POINTS_PER_HOUR = 3600 * POINTS_PER_SECOND 

print(f"Frequência de Amostragem Detectada: {POINTS_PER_SECOND} Hz (0.1s)")
print(f"Divisor de Conversão (Pontos -> Horas): {POINTS_PER_HOUR}\n")

# ====================================================================
# 1. ANÁLISE POR FROTA (TRATOR) - EM HORAS CORRIGIDAS
# ====================================================================
print("## 1. Estatística de Frequência da Frota (Horas de Operação)\n")

# Contagem
df_frota = full_df['tractor'].value_counts().reset_index()
df_frota.columns = ['Trator', 'Total Datapoints']

# Conversão para Horas (USANDO O NOVO DIVISOR)
df_frota['Horas Totais (h)'] = df_frota['Total Datapoints'] / POINTS_PER_HOUR
df_frota['Proporção (%)'] = (df_frota['Horas Totais (h)'] / df_frota['Horas Totais (h)'].sum()) * 100

# Formatação
df_frota_print = df_frota.copy()
df_frota_print['Horas Totais (h)'] = df_frota_print['Horas Totais (h)'].map('{:,.1f}'.format)
df_frota_print['Proporção (%)'] = df_frota_print['Proporção (%)'].round(2)

print(df_frota_print[['Trator', 'Horas Totais (h)', 'Proporção (%)']].to_string(index=False))
print(f"\nTotal de Horas da Frota no Dataset: {df_frota['Horas Totais (h)'].sum():,.1f} h")
print("\n" + "="*80 + "\n")


# ====================================================================
# 2. ANÁLISE DO RÓTULO DE ATIVIDADE - EM HORAS CORRIGIDAS
# ====================================================================
print("## 2. Distribuição das Atividades (Ground Truth em Horas)\n")

df_atividade = full_df['activity'].value_counts().reset_index()
df_atividade.columns = ['Atividade', 'Total Datapoints']

# Conversão para Horas (USANDO O NOVO DIVISOR)
df_atividade['Horas Totais (h)'] = df_atividade['Total Datapoints'] / POINTS_PER_HOUR
df_atividade['Proporção (%)'] = (df_atividade['Horas Totais (h)'] / df_atividade['Horas Totais (h)'].sum()) * 100

# Formatação
df_act_print = df_atividade.copy()
df_act_print['Horas Totais (h)'] = df_act_print['Horas Totais (h)'].map('{:,.1f}'.format)
df_act_print['Proporção (%)'] = df_act_print['Proporção (%)'].round(2)

print(df_act_print[['Atividade', 'Horas Totais (h)', 'Proporção (%)']].to_string(index=False))

# Gráfico 1: Barras
fig_act = px.bar(df_atividade, 
                 x='Atividade', y='Horas Totais (h)', 
                 title='Distribuição Total de Horas por Atividade Rotulada (10Hz Corrected)',
                 color='Horas Totais (h)', 
                 color_continuous_scale=px.colors.sequential.Viridis)
fig_act.show()
print("\n" + "="*80 + "\n")


# ====================================================================
# OTIMIZAÇÃO PARA BOXPLOTS (Amostragem)
# ====================================================================
print("## Preparando Amostra Estatística para Boxplots...")
SAMPLE_SIZE = 100000 

if len(full_df) > SAMPLE_SIZE:
    # Amostragem aleatória simples mantém a distribuição estatística
    df_sample = full_df.sample(n=SAMPLE_SIZE, random_state=42).copy()
else:
    df_sample = full_df.copy()

# Garantir tipos numéricos
df_sample['fuel_rate'] = pd.to_numeric(df_sample['fuel_rate'], errors='coerce')
df_sample['veh_speed'] = pd.to_numeric(df_sample['veh_speed'], errors='coerce')

# Top 10 atividades para limpar o gráfico
top_10_activities = df_atividade['Atividade'].head(10).tolist()
df_viz = df_sample[df_sample['activity'].isin(top_10_activities)]

print("Amostra pronta. Gerando gráficos...\n")


# ====================================================================
# 3. BOXPLOT DE CONSUMO
# ====================================================================
print("## 3. Boxplot Discriminativo: Consumo por Atividade\n")

fig_box = px.box(df_viz, 
                 x='activity', 
                 y='fuel_rate',
                 color='activity',
                 title=f'Distribuição de Consumo (L/h) - Amostra {SAMPLE_SIZE}',
                 points="outliers" 
                )
limit_y = df_viz['fuel_rate'].quantile(0.995) 
fig_box.update_yaxes(range=[0, limit_y]) 
fig_box.show()

print("\n" + "="*80 + "\n")


# ====================================================================
# 4. BOXPLOT DE VELOCIDADE
# ====================================================================
print("## 4. Boxplot Discriminativo: Velocidade por Atividade\n")

fig_speed = px.box(df_viz, 
                 x='activity', 
                 y='veh_speed',
                 color='activity',
                 title=f'Distribuição de Velocidade (m/s) - Amostra {SAMPLE_SIZE}',
                 points="outliers"
                )
limit_speed_y = df_viz['veh_speed'].quantile(0.995)
fig_speed.update_yaxes(range=[0, limit_speed_y])
fig_speed.show()

In [ ]:
# ==============================================================================
# ANÁLISE ESTATÍSTICA: TRATORES (FINAL - Escala de Cinza & Visual Limpo)
# ==============================================================================
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc # Import para manipular cores
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- FUNÇÃO AUXILIAR: LETRAS DE SIGNIFICÂNCIA (CLD) ---
def get_letters(tukey_results, means_sorted):
    """Gera letras (a, b, c) para diferenciar grupos estatísticos."""
    labels = means_sorted.index.tolist()
    letters = {l: '' for l in labels}
    df_res = pd.DataFrame(data=tukey_results.summary().data[1:], columns=tukey_results.summary().data[0])
    
    current_letter_idx = 0
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    not_assigned = labels.copy()
    
    while not_assigned:
        base = not_assigned[0]
        letter = alphabet[current_letter_idx]
        letters[base] += letter
        
        # Encontra quem é estatisticamente IGUAL ao base
        for other in not_assigned[1:]:
            pair = df_res[((df_res['group1'] == base) & (df_res['group2'] == other)) | 
                          ((df_res['group1'] == other) & (df_res['group2'] == base))]
            if not pair.empty and not pair['reject'].values[0]: # reject=False (Igual)
                 letters[other] += letter
        
        not_assigned.pop(0)
        current_letter_idx += 1
        
    return letters

# --- CONFIGURAÇÃO ---
SAMPLE_SIZE_ANOVA = 500000 

print(f"--- INICIANDO ANÁLISE DE MÉDIAS (TRATORES) ---\n")

if 'full_df' in locals():
    # 1. Filtro e Amostragem
    df_stats = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()
    
    if len(df_stats) > SAMPLE_SIZE_ANOVA:
        df_sample = df_stats.groupby('tractor', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(SAMPLE_SIZE_ANOVA * len(x)/len(df_stats))), random_state=42)
        )
    else:
        df_sample = df_stats.copy()

    # 2. ANOVA
    grupos = [d['fuel_rate'].values for name, d in df_sample.groupby('tractor')]
    f_stat, p_value = stats.f_oneway(*grupos)
    print(f"ANOVA F-value: {f_stat:.2f} | P-value: {p_value:.4e}")

    if p_value < 0.05:
        # 3. Tukey e Letras
        tukey = pairwise_tukeyhsd(endog=df_sample['fuel_rate'], groups=df_sample['tractor'], alpha=0.05)
        summary = df_sample.groupby('tractor')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        
        # Garante que a função get_letters existe
        if 'get_letters' in locals():
            letter_dict = get_letters(tukey, summary['mean'])
            summary['Letra'] = summary.index.map(letter_dict)
        else:
            summary['Letra'] = '?'
        
        # 4. Tabela Final
        print("\n=== CONSUMO MÉDIO POR TRATOR (Letras diferentes = Diferença significativa) ===")
        print(f"{'Trator':<15} | {'Média ± DP (L/h)':<20} | {'Grupo'}")
        print("-" * 55)
        for trator, row in summary.iterrows():
            print(f"{trator:<15} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Letra']}")
        print("-" * 55)

        # 5. CONFIGURAÇÃO DE CORES (Preto -> Cinza Claro)
        n_colors = len(summary)
        # Gera escala do Preto (1.0) até Cinza Claro (0.25)
        sample_points = np.linspace(1.0, 0.25, n_colors) 
        custom_greys = pc.sample_colorscale('Greys', sample_points)

        # 6. GRÁFICO BOXPLOT
        fig = px.box(
            df_sample, 
            x='tractor', 
            y='fuel_rate',
            color='tractor',
            category_orders={'tractor': summary.index.tolist()}, # Ordena do Maior para Menor
            color_discrete_sequence=custom_greys, # Nossa escala personalizada
            title=f"Distribuição de Consumo por Trator (ANOVA p < 0.0001)",
            labels={'fuel_rate': 'Consumo de Combustível (Litros/hora)', 'tractor': 'Modelo'},
            points=False # Remove pontos para limpar visual
        )
        
        # Adiciona média (Diamante Vermelho para contraste)
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Média'
        ))
        
        fig.update_layout(
            showlegend=False, 
            height=500,
            template='plotly_white', # Fundo branco limpo
            font=dict(color="black")
        )
        fig.show()
        
    else:
        print("Sem diferenças significativas.")
else:
    print("ERRO: full_df não carregado.")

In [ ]:
# ==============================================================================
# ANÁLISE ESTATÍSTICA: ATIVIDADES (FINAL - Escala de Cinza Personalizada)
# ==============================================================================
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc # Import para manipular cores
import scipy.stats as stats
import numpy as np # Import necessário para gerar a escala numérica
from statsmodels.stats.multicomp import pairwise_tukeyhsd

SAMPLE_SIZE = 500000
TOP_N = 12 

print(f"--- INICIANDO ANÁLISE DE MÉDIAS (ATIVIDADES) ---\n")

if 'full_df' in locals():
    # 1. Filtros
    df_clean = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()
    top_activities = df_clean['activity'].value_counts().head(TOP_N).index.tolist()
    df_clean = df_clean[df_clean['activity'].isin(top_activities)]
    
    # 2. Amostragem
    if len(df_clean) > SAMPLE_SIZE:
        frac = SAMPLE_SIZE / len(df_clean)
        df_sample = df_clean.groupby('activity', group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=42)
        )
    else:
        df_sample = df_clean.copy()

    # --- TRADUÇÃO PRÉVIA ---
    traducoes_display = {
        'Cultivating (deep)': 'Cultivo Profundo', 
        'Cultivating (shallow)': 'Cultivo Superficial',
        'Disc harrowing': 'Gradeação', 
        'Fertilizing': 'Adubação', 
        'Mowing (front)': 'Ceifa Frontal',
        'Mowing (large-scale)': 'Ceifa Larga', 
        'Mulching': 'Trituração', 
        'Ploughing': 'Aração',
        'Power harrowing': 'Grade Rotativa', 
        'Seed drill combination': 'Semeadura Conj.',
        'Seedbed combination': 'Prep. Leito', 
        'Spraying': 'Pulverização', 
        'Transport': 'Transporte',
        'not working': 'Ocioso/Parado'
    }
    
    # Cria coluna traduzida
    df_sample['Atividade_PT'] = df_sample['activity'].map(traducoes_display).fillna(df_sample['activity'])

    # 3. ANOVA
    grupos = [d['fuel_rate'].values for name, d in df_sample.groupby('activity')]
    f_stat, p_value = stats.f_oneway(*grupos)
    print(f"ANOVA F-value: {f_stat:.2f} | P-value: {p_value:.4e}")

    if p_value < 0.05:
        # 4. Tukey 
        tukey = pairwise_tukeyhsd(endog=df_sample['fuel_rate'], groups=df_sample['activity'], alpha=0.05)
        
        # Agrupa e Ordena (Do Maior para o Menor)
        summary = df_sample.groupby('Atividade_PT')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        
        # Letras (Se a função existir)
        if 'get_letters' in locals():
            summary_en = df_sample.groupby('activity')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
            letter_dict_en = get_letters(tukey, summary_en['mean'])
            map_en_pt = {k: traducoes_display.get(k, k) for k in letter_dict_en.keys()}
            letter_dict_pt = {map_en_pt[k]: v for k, v in letter_dict_en.items()}
            summary['Letra'] = summary.index.map(letter_dict_pt)
        else:
            summary['Letra'] = '?'

        print("\n=== CONSUMO MÉDIO POR ATIVIDADE ===")
        print(f"{'Atividade':<25} | {'Média ± DP (L/h)':<20} | {'Grupo'}")
        print("-" * 65)
        for idx, row in summary.iterrows():
            print(f"{idx:<25} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Letra']}")
        print("-" * 65)

        # 5. CONFIGURAÇÃO DE CORES PERSONALIZADA (Preto -> Cinza Claro, sem Branco)
        # Quantas cores precisamos?
        n_colors = len(summary)
        
        # Geramos uma escala numérica de 0 (Preto) a 1 (Branco), mas paramos em 0.8 (Cinza Claro)
        # O 'Greys' do Plotly vai do Branco(0) ao Preto(1). Queremos o inverso.
        # Vamos amostrar do 1.0 (Preto) até 0.2 (Cinza Escuro/Médio). Evitamos o 0.0 (Branco).
        sample_points = np.linspace(1.0, 0.25, n_colors) 
        custom_greys = pc.sample_colorscale('Greys', sample_points)

        # 6. GRÁFICO BOXPLOT
        order_idx_pt = summary.index.tolist() 
        
        fig = px.box(
            df_sample, 
            x='Atividade_PT', 
            y='fuel_rate',
            category_orders={'Atividade_PT': order_idx_pt}, 
            color='Atividade_PT',
            color_discrete_sequence=custom_greys, # Aplica nossa escala manual
            title=f"Dispersão de Consumo por Atividade (ANOVA p < 0.0001)",
            labels={'fuel_rate': 'Consumo de Combustível (Litros/hora)', 'Atividade_PT': 'Operação'},
            points=False
        )
        
        # Adiciona média (Diamante Vermelho)
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Média'
        ))
        
        fig.update_layout(
            showlegend=False, 
            height=600,
            template='plotly_white', 
            font=dict(color="black")
        )
        fig.show()
        
    else:
        print("Sem diferenças significativas.")
else:
    print("ERRO: full_df não carregado.")

In [ ]:
# ==============================================================================
# ANÁLISE DE OCIOSIDADE POR TRATOR (Tempo Parado com Motor Ligado)
# ==============================================================================

# --- PARÂMETROS DE DEFINIÇÃO DE "OCIO" ---
THRESHOLD_SPEED_MS = 0.75  # < 0.75 m/s (aprox 2.7 km/h) é considerado parado/manobra
THRESHOLD_RPM = 100        # > 100 RPM considera o motor ligado
FREQ_HZ = 10               # Frequência dos dados (10Hz)
POINTS_PER_HOUR = 3600 * FREQ_HZ

print(f"--- ANÁLISE DE OCIOSIDADE (Critério: Vel < {THRESHOLD_SPEED_MS} m/s e RPM > {THRESHOLD_RPM}) ---\n")

if 'full_df' in locals():
    # 1. Filtra apenas momentos com motor ligado (ignoramos trator desligado)
    df_engine_on = full_df[full_df['engine_rpm'] > THRESHOLD_RPM].copy()
    
    # 2. Cria a flag de Ociosidade (Booleano)
    # True se estiver lento/parado, False se estiver andando
    df_engine_on['is_idle'] = df_engine_on['veh_speed'] < THRESHOLD_SPEED_MS
    
    # 3. Agrupamento por Trator
    # 'count' conta o total de linhas (tempo total ligado)
    # 'sum' na coluna booleana conta quantos True (tempo ocioso)
    stats_tractor = df_engine_on.groupby('tractor')['is_idle'].agg(['count', 'sum']).reset_index()
    stats_tractor.columns = ['Trator', 'Total Pontos', 'Pontos Idle']
    
    # 4. Conversão para Horas e Porcentagem
    stats_tractor['Horas Totais (h)'] = stats_tractor['Total Pontos'] / POINTS_PER_HOUR
    stats_tractor['Horas Ociosas (h)'] = stats_tractor['Pontos Idle'] / POINTS_PER_HOUR
    stats_tractor['% Ocioso'] = (stats_tractor['Horas Ociosas (h)'] / stats_tractor['Horas Totais (h)']) * 100
    
    # 5. Formatação e Exibição
    stats_tractor = stats_tractor.sort_values('% Ocioso', ascending=False) # Do mais ocioso para o menos
    
    print("=== RANKING DE OCIOSIDADE POR TRATOR ===")
    print(f"{'Trator':<15} | {'Horas Ligado':<15} | {'Horas Parado':<15} | {'% Ocioso'}")
    print("-" * 65)
    
    for _, row in stats_tractor.iterrows():
        print(f"{row['Trator']:<15} | {row['Horas Totais (h)']:<15.1f} | {row['Horas Ociosas (h)']:<15.1f} | {row['% Ocioso']:.1f}%")
    print("-" * 65)
    print(f"Total de Horas Analisadas (Motor On): {stats_tractor['Horas Totais (h)'].sum():,.1f} h")

else:
    print("ERRO: full_df não carregado.")

In [ ]:
# ==============================================================================
# ANÁLISE DE OCIOSIDADE POR ATIVIDADE (Onde o tempo é perdido?)
# ==============================================================================

print(f"--- DETALHAMENTO DE OCIOSIDADE DENTRO DAS ATIVIDADES ---\n")

if 'full_df' in locals() and 'df_engine_on' in locals():
    # Usamos o df_engine_on criado na célula anterior para garantir consistência
    
    # 1. Agrupamento por Atividade
    stats_activity = df_engine_on.groupby('activity')['is_idle'].agg(['count', 'sum']).reset_index()
    stats_activity.columns = ['Atividade', 'Total Pontos', 'Pontos Idle']
    
    # 2. Conversão
    stats_activity['Horas Totais (h)'] = stats_activity['Total Pontos'] / POINTS_PER_HOUR
    stats_activity['Horas Ociosas (h)'] = stats_activity['Pontos Idle'] / POINTS_PER_HOUR
    stats_activity['% Ocioso'] = (stats_activity['Horas Ociosas (h)'] / stats_activity['Horas Totais (h)']) * 100
    
    # 3. Filtro e Ordenação
    # Filtramos atividades muito curtas (< 10h totais) para não poluir a tabela com ruído
    stats_activity = stats_activity[stats_activity['Horas Totais (h)'] > 10].sort_values('% Ocioso', ascending=False)
    
    # 4. Tradução para Exibição (Opcional, mas recomendado)
    traducoes = {
        'Transport': 'Transporte', 'Seed drill combination': 'Semeadura', 
        'Ploughing': 'Aração', 'not working': 'Não Trabalhando (Geral)',
        'Spraying': 'Pulverização', 'Fertilizing': 'Adubação',
        'Cultivating (deep)': 'Cultivo Profundo', 'Harvesting': 'Colheita'
    }
    stats_activity['Nome PT'] = stats_activity['Atividade'].map(traducoes).fillna(stats_activity['Atividade'])
    
    print("=== OCIOSIDADE INTRÍNSECA POR ATIVIDADE (Top Lista) ===")
    print(f"{'Atividade':<25} | {'Duração Total':<15} | {'Tempo Parado':<15} | {'% Ocioso'}")
    print("-" * 75)
    
    for _, row in stats_activity.iterrows():
        # Destaque visual para ociosidade alta (>30%)
        alerta = "(!)" if row['% Ocioso'] > 30 else ""
        print(f"{row['Nome PT']:<25} | {row['Horas Totais (h)']:<15.1f} | {row['Horas Ociosas (h)']:<15.1f} | {row['% Ocioso']:.1f}% {alerta}")
        
    print("-" * 75)
    print("* Considerado 'Parado' se Velocidade < 1 m/s enquanto Motor Ligado.")
    print("* Atividades com menos de 10h de duração foram ocultadas.")

else:
    print("ERRO: Rode a célula anterior primeiro para gerar 'df_engine_on'.")